# 18.11 教師的風格和錯誤會被學走嗎？


學生回答越來越像教師，可能也把教師的錯誤一起學走。蒸餾資料裡不僅有內容，還有篇幅、語氣、格式與「什麼時候說不知道」等行為。如果教師每題都說不知道，學生模仿得很成功，仍可能無法完成原本能回答的問題。前置是[獨立真值評估](https://birdhackor.github.io/tiny-perceptron-vlm/18.10.html)與[不知道時怎麼回答](https://birdhackor.github.io/tiny-perceptron-vlm/9.2.html)；我們要將「像教師」與「符合任務」分別量。

取三個問題：「1+1=?」「2+2=?」「我的晚餐是什麼？」。前兩題有答案2與4，最後一題沒有提供個人資訊，應揭示不足。教師三題都回答「不知道」，學生完全複製它。教師一致率是100%，但前兩題沒有完成；第三題則合理。若評估只做字串exact match，把第三題標準答案寫成「資訊不足」，又會錯把意思合適的「不知道」判成錯誤。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
teacher_answers = ["不知道", "不知道", "不知道"]
student_answers = teacher_answers.copy()
truth = ["2", "4", None]


def acceptable(answer, expected):
    if expected is None:
        return "不知道" in answer or "資訊不足" in answer
    return answer.strip() == expected


agreement = sum(a == b for a, b in zip(teacher_answers, student_answers)) / 3
checks = [acceptable(a, t) for a, t in zip(student_answers, truth)]
print("教師一致率", agreement)
print("逐題合格", checks)
print("任務合格率", round(sum(checks) / 3, 4))

這裡用人工回答清單，沒有模型生成。truth中的None表示第三題不應捏造確定答案，acceptable按本題規則接受兩種揭示不足措辭；前兩題則按已知算術答案核對。`zip`將相同位置配成一對，`sum`把True視為1、False為0。輸出一致率1.0，逐題 `[False,False,True]`，任務合格率0.3333。這個很小的判準不是所有開放題的通用判分器，只為示範獨立條件如何影響結論。

風格也可能傳遞，例如教師習慣每題長篇解釋，學生會學到篇幅，即使用戶要求簡短。要看內容、格式、長度與完成情況，而不是只有文字相似度。安全與拒絕行為同樣需分開檢查：該保留的回應限制不能因為模仿文字而消失，普通可回答任務也不能因教師過度拒絕而全部跳過，回顧[過度拒絕](https://birdhackor.github.io/tiny-perceptron-vlm/9.6.html)。

資料生成後應檢查已知錯誤、不能驗證的資訊與偏離目標的風格，保存過濾原因。這不保證學生不犯錯，但讓你能追到它看過哪些訊號。對教師本身與SFT學生也用同樣判準，才知道蒸餾究竟繼承了什麼、又在哪些地方改善或退步。

練習在最初的teacher_answers清單那行，只把第一項字串`"不知道"`改成字串`"2"`，保留引號；在它之後的student_answers仍照copy取得回答。回答在本例都是文字，改成整數2會讓後面的`.strip()`報錯。先預測一致率依然1.0，逐題變 `[True,False,True]`、合格率0.6667，再執行核對。改進來自教師內容更正，而不是一致率更高；這兩個指標不能互相替代。
